In [ ]:
import numpy as np
import itertools
from collections import Counter


# Simple Neural Network class
class SimpleNN:
    def _init_(self, input_size, hidden_size, output_size):
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.relu(self.Z1)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.softmax(self.Z2)
        return self.A2

    def relu(self, Z):
        return np.maximum(0, Z)

    def softmax(self, Z):
        expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))  # Stability improvement
        return expZ / expZ.sum(axis=1, keepdims=True)

    def compute_loss(self, Y, Y_hat):
        m = Y.shape[0]
        return -np.sum(Y * np.log(Y_hat + 1e-9)) / m

    def backward(self, X, Y):
        m = Y.shape[0]
        dZ2 = self.A2 - Y
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * (self.A1 > 0)
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # Update weights and biases
        self.W1 -= 0.01 * dW1
        self.b1 -= 0.01 * db1
        self.W2 -= 0.01 * dW2
        self.b2 -= 0.01 * db2


# Function to one-hot encode the labels
def one_hot_encode(labels, num_classes):
    one_hot = np.zeros((len(labels), num_classes))
    one_hot[np.arange(len(labels)), labels] = 1
    return one_hot


# Load corpus function
def load_corpus(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()


# Generate permutations of characters
def generate_permutations(chars, max_length):
    permuted_tokens = set()
    for length in range(1, max_length + 1):
        for p in itertools.permutations(chars, length):
            permuted_tokens.add("".join(p))
    return permuted_tokens


# Count frequencies of permuted tokens in the corpus
def count_permuted_frequencies(corpus, permuted_tokens):
    frequencies = Counter()
    for token in permuted_tokens:
        frequencies[token] += corpus.count(token)
    return frequencies


# Get the top tokens by frequency
def get_top_tokens(frequencies, max_tokens):
    return frequencies.most_common(max_tokens)


# Prepare dataset from the corpus
def prepare_dataset(corpus, tokens):
    token_to_index = {token: idx for idx, token in enumerate(tokens)}
    X = []
    Y = []
    for i in range(len(corpus) - 1):
        if corpus[i] in token_to_index and corpus[i + 1] in token_to_index:
            X.append(token_to_index[corpus[i]])
            Y.append(token_to_index[corpus[i + 1]])
    return np.array(X), np.array(Y)


# Text generation function
def generate_text(model, start_text, tokens, length=50):
    token_to_index = {token: idx for idx, token in enumerate(tokens)}
    index_to_token = {idx: token for idx, token in enumerate(tokens)}

    generated = start_text
    current_input = [
        token_to_index[char] for char in start_text if char in token_to_index
    ]

    for _ in range(length):
        if not current_input:
            break

        X_one_hot = np.zeros((1, len(tokens)))
        X_one_hot[0, current_input[-1]] = 1

        Y_hat = model.forward(X_one_hot)
        next_token_idx = np.random.choice(len(tokens), p=Y_hat.flatten())
        next_token = index_to_token[next_token_idx]

        generated += next_token
        current_input.append(next_token_idx)

    return generated


# Main execution
def main(file_path, max_length, max_tokens):
    corpus = load_corpus(file_path)

    # Get unique characters from the corpus
    unique_chars = set(corpus)

    # Generate permutations
    permuted_tokens = generate_permutations(unique_chars, max_length)

    # Count frequencies of each permuted token in the corpus
    frequencies = count_permuted_frequencies(corpus, permuted_tokens)

    # Get the top tokens
    top_tokens = get_top_tokens(frequencies, max_tokens)

    # Extract tokens and frequencies
    tokens = [token for token, _ in top_tokens]

    # Prepare the dataset
    X, Y = prepare_dataset(corpus, tokens)

    # Check if dataset is empty
    if len(X) == 0 or len(Y) == 0:
        print(
            "No valid data to train the model. Please check your corpus and token generation."
        )
        return

    # One-hot encode the input and output
    X_one_hot = np.zeros((len(X), len(tokens)))
    X_one_hot[np.arange(len(X)), X] = 1

    # One-hot encode Y
    Y_one_hot = one_hot_encode(Y, len(tokens))

    # Initialize the SimpleNN
    input_size = len(tokens)
    hidden_size = 16
    output_size = len(tokens)

    nn = SimpleNN(
        input_size=input_size, hidden_size=hidden_size, output_size=output_size
    )

    # Training loop for SimpleNN
    for epoch in range(1000):
        Y_hat = nn.forward(X_one_hot)
        loss = nn.compute_loss(Y_one_hot, Y_hat)

        nn.backward(X_one_hot, Y_one_hot)

        if epoch % 100 == 0:
            print(f"SimpleNN - Epoch {epoch}, Loss: {loss:.4f}")

    # Generate text from the trained model
    initial_text = "Additive "  # Provide some initial text
    generated_text = generate_text(nn, initial_text, tokens, length=50)
    print("Generated Text:")
    print(generated_text)


# Example usage
file_path = "input.txt"  # Path to your input file
max_length = 4  # Max token size
max_tokens = 600  # Number of top tokens to retrieve

main(file_path, max_length, max_tokens)